In [1]:
!pip install -q torch transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.2 MB/s eta 0:00:00


In [1]:
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


GPU available: True
Device: Tesla T4


In [22]:
MODEL_NAME       = "meta-llama/Llama-3.2-1B"
TRAIN_SAMPLES    = 3000
EVAL_SAMPLES     = 1000
MAX_SEQ_LEN      = 512
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
OUTPUT_DIR       = "./gsm8k_lora"
NUM_EPOCHS       = 3
BATCH_SIZE       = 4
GRAD_ACCUM       = 8
LEARNING_RATE    = 2e-4


In [23]:
from huggingface_hub import login
login()


In [24]:
dataset = load_dataset("openai/gsm8k", "main")

train_data = dataset["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
eval_data  = dataset["test"].shuffle(seed=42).select(range(EVAL_SAMPLES))

print(f"Train samples : {len(train_data)}")
print(f"Eval samples  : {len(eval_data)}")

print("\n── Sample ──")
print("Question:", train_data[0]["question"])
print("Answer  :", train_data[0]["answer"])


Train samples : 3000
Eval samples  : 1000

── Sample ──
Question: Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?
Answer  : Mimi has 2 x 12 = <<2*12=24>>24 sea shells.
Kyle has 24 x 2 = <<24*2=48>>48 sea shells.
Leigh has 48 / 3 = <<48/3=16>>16 sea shells.
#### 16


In [25]:
PROMPT_TEMPLATE = """### Question:
{question}

### Solution:
{answer}"""

INFERENCE_TEMPLATE = """### Question:
{question}

### Solution:
"""

def format_sample(sample):
    return PROMPT_TEMPLATE.format(
        question=sample["question"].strip(),
        answer=sample["answer"].strip()
    )

print(format_sample(train_data[0]))


### Question:
Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?

### Solution:
Mimi has 2 x 12 = <<2*12=24>>24 sea shells.
Kyle has 24 x 2 = <<24*2=48>>48 sea shells.
Leigh has 48 / 3 = <<48/3=16>>16 sea shells.
#### 16


In [26]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size : {tokenizer.vocab_size}")
print(f"Pad token  : {tokenizer.pad_token}")


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Vocab size : 128000
Pad token  : <|end_of_text|>


In [27]:
class GSM8KDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=MAX_SEQ_LEN):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample    = self.data[idx]
        full_text = format_sample(sample)

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids      = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        prompt_text = INFERENCE_TEMPLATE.format(question=sample["question"].strip())
        prompt_len  = len(self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

        labels          = input_ids.clone()
        labels[:prompt_len] = -100

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
        }

train_dataset = GSM8KDataset(train_data, tokenizer)
eval_dataset  = GSM8KDataset(eval_data,  tokenizer)

print(f"Train dataset size : {len(train_dataset)}")
print(f"Eval dataset size  : {len(eval_dataset)}")


Train dataset size : 3000
Eval dataset size  : 1000


In [28]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type       = TaskType.CAUSAL_LM,
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    target_modules  = ["q_proj", "v_proj"],
    bias            = "none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [29]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 0.05,
    weight_decay                = 0.01,
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "steps",
    eval_steps                  = 200,
    save_steps                  = 200,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    report_to                   = "none",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    processing_class = tokenizer,
    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8
    ),
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss,Validation Loss
200,0.151716,0.160942


TrainOutput(global_step=282, training_loss=0.4372415491875182, metrics={'train_runtime': 1554.5879, 'train_samples_per_second': 5.789, 'train_steps_per_second': 0.181, 'total_flos': 2.6952654127104e+16, 'train_loss': 0.4372415491875182, 'epoch': 3.0})

In [30]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")


Model saved to: ./gsm8k_lora


In [31]:
def extract_answer(text):
    match = re.search(r"####\s*([\d,]+\.?\d*)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    numbers = re.findall(r'\d+\.?\d*', text)
    return numbers[-1] if numbers else None


def evaluate(model, tokenizer, test_data, n_samples=100):
    model.eval()
    correct = 0
    total   = 0

    for i, sample in enumerate(test_data):
        if i >= n_samples:
            break

        prompt = INFERENCE_TEMPLATE.format(question=sample["question"].strip())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens  = 256,
                do_sample       = False,
                pad_token_id    = tokenizer.eos_token_id,
            )

        generated       = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predicted       = extract_answer(generated)
        ground_truth    = extract_answer(sample["answer"])

        if predicted and ground_truth and predicted == ground_truth:
            correct += 1
        total += 1

        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{n_samples}] Accuracy so far: {correct/total:.3f}")

    print(f"\nFinal Exact Match Accuracy: {correct/total:.4f} ({correct}/{total})")
    return correct / total


accuracy = evaluate(model, tokenizer, eval_data, n_samples=100)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [20/100] Accuracy so far: 0.200
  [40/100] Accuracy so far: 0.125
  [60/100] Accuracy so far: 0.167
  [80/100] Accuracy so far: 0.150
  [100/100] Accuracy so far: 0.130

Final Exact Match Accuracy: 0.1300 (13/100)


In [32]:
sample  = eval_data[0]
prompt  = INFERENCE_TEMPLATE.format(question=sample["question"].strip())
inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens = 256,
        do_sample      = False,
        pad_token_id   = tokenizer.eos_token_id,
    )

print("── Question ──")
print(sample["question"])
print("\n── Ground Truth ──")
print(sample["answer"])
print("\n── Model Output ──")
print(tokenizer.decode(output[0], skip_special_tokens=True))


── Question ──
Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

── Ground Truth ──
The total ratio representing their ages is 7+11= <<7+11=18>>18
Since the fraction of the ratio that represents Allen's age is 11/18, Allen's current age is 11/18*162 = <<11/18*162=99>>99
If Allen is currently 99 years old, in 10 years he will be 99+10 = <<99+10=109>>109 years old
#### 109

── Model Output ──
### Question:
Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

### Solution:
The ratio of their ages is 7:11, so their ages are 7*11 = <<7*11=77>>77 years old.
If their total age is 162, then their ages 10 years from now will be 77+10 = <<77+10=87>>87 years old.
Allen's age 10 years from now is 87-10 = <<87-10=77>>77 years old.
#### 77
